# LLM Pipeline Test — Step 6: Tool Optimization

The **Optimizer** is decoupled from gameplay.  This notebook orchestrates:

1. **Play** a Sokoban game with the MCTS engine → generate trace
2. **Collect** tool sources from the engine
3. **Optimise** — feed traces + tool sources to the Optimizer (LLM analysis → code generation → parse / validate / install → smoke test)
4. **Evaluate** — hot-swap the returned function into a fresh engine

We use **level3** (2 boxes) since default MCTS cannot solve it — the ideal test case for LLM improvement.

## Cell 1: Setup & imports

In [1]:
import sys
from pathlib import Path

# Ensure Tool_Creation is on sys.path
ROOT = Path(".").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from mcts import MCTSEngine
from mcts.games import Sokoban
from LLM import Optimizer

# -- Configuration (single place to tweak) --
GAME      = "sokoban"
LEVELS = ["level1", "level2", "level3", "level4", "level5", "level6", "level7", "level8", "level9", "level10"]
LEVEL = "level3"
PHASE     = "simulation"        # MCTS phase to improve
ITERS     = 200                 # MCTS iterations per move
MAX_DEPTH = 500                 # max rollout depth
MAX_STEPS = 200                 # max game steps

print(f"Working dir: {ROOT}")
print("All imports OK ✓")

Working dir: /Users/zhuyuezx/Documents/UCSD/Winter_2025/CSE291A/CSE291A_Project/Tool_Creation
All imports OK ✓


## Cell 2: Play baseline game → collect trace & tool sources

The engine produces the trace file and exposes the current tool sources.
The Optimizer never touches the engine.

In [2]:
# Play one baseline game
game = Sokoban(LEVEL, max_steps=MAX_STEPS)
engine = MCTSEngine(game, iterations=ITERS, max_rollout_depth=MAX_DEPTH, logging=True)
baseline = engine.play_game()

# Collect outputs for the optimizer
trace_file  = baseline.get("log_file", "")
tool_list   = engine.get_tool_source()          # dict[str, str]

base_tag = "SOLVED" if baseline.get("solved") else "UNSOLVED"
print(f"Baseline: {base_tag} in {baseline.get('steps', '?')} steps  "
      f"returns={baseline.get('returns', '?')}")
print(f"Trace: {trace_file}")
print(f"Tools: {list(tool_list.keys())}")

Baseline: SOLVED in 82 steps  returns=[1.0]
Trace: /Users/zhuyuezx/Documents/UCSD/Winter_2025/CSE291A/CSE291A_Project/Tool_Creation/mcts/records/Sokoban (level3)_20260306_030346_095666.json
Tools: ['selection', 'expansion', 'simulation', 'backpropagation']


## Cell 3: Iterative Optimization Loop

Each iteration:
1. **Pick a random level** → play with current tool → new trace
2. **Build history context** from last 3 iterations
3. **Run optimizer** (3-step: analysis → draft → light critique)
4. **Evaluate** the new tool on the **same level** with 3 runs
5. **Accept or reject** — compare against that level's **own baseline**

Key design:
- **Per-level baselines**: each level has its own baseline composite score, computed lazily on first encounter
- **Accept-unless-terrible**: accept if composite ≥ 50% of that level's baseline. Rejects only catastrophic regressions, lets the LLM iterate freely.
- **`current_fn`** always advances on acceptance, **`best_fn`** tracks the function with best **average composite across all evaluated levels**
- **Incremental prompting**: LLM makes ONE small change per iteration
- **History window**: only last 3 iterations shown to LLM (avoids context bloat)

In [3]:
import time, random

NUM_ITERS    = 5
EVAL_RUNS    = 3          # average over this many games per evaluation
SOLVE_WEIGHT = 0.6        # composite = SOLVE_WEIGHT * solve_rate + RETURN_WEIGHT * avg_returns
RETURN_WEIGHT = 0.4
REJECT_THRESHOLD = 0.5    # reject only if composite < baseline * this factor
MASTERY_SOLVE_RATE = 1.0  # levels with solve_rate ≥ this are considered mastered
MASTERY_MAX_STEPS  = None # optional: also require avg_steps ≤ this (None = ignore steps)

opt = Optimizer(
    game=GAME,
    target_phase=PHASE,
    three_step=True,        # analysis → draft → critique+finalize
    verbose=True,
)

# State that carries across iterations
best_fn       = None       # best-ever function (None = default)
current_fn    = None       # latest accepted function (feeds next iteration)
all_results   = []
history_window = 3         # how many past iterations to include in the LLM context

# --- Per-level tracking ---
# Baselines are computed lazily on first encounter of each level.
level_baselines: dict[str, dict] = {}   # level → {composite, avg_returns, solve_rate, ...}
level_best_scores: dict[str, float] = {}  # level → best composite seen on that level
mastered_levels: set[str] = set()         # levels already solved at 100% — skip these
active_levels = list(LEVELS)              # mutable pool for random selection


def composite_score(solve_rate, avg_returns):
    """Weighted score that prioritizes solving over raw returns."""
    return SOLVE_WEIGHT * solve_rate + RETURN_WEIGHT * avg_returns


def check_mastery(level, solve_rate, avg_steps):
    """Mark a level as mastered if it meets the mastery criteria."""
    if level in mastered_levels:
        return
    if solve_rate < MASTERY_SOLVE_RATE:
        return
    if MASTERY_MAX_STEPS is not None and avg_steps > MASTERY_MAX_STEPS:
        return
    mastered_levels.add(level)
    if level in active_levels:
        active_levels.remove(level)
    steps_info = f", avg_steps={avg_steps:.0f}" if avg_steps is not None else ""
    print(f"  🎓 {level} MASTERED (solve_rate={solve_rate:.0%}{steps_info}) "
          f"— removed from pool. {len(active_levels)} levels remain.")


def multi_eval(fn, level, n=EVAL_RUNS):
    """Evaluate a function over n games on a specific level."""
    t0 = time.time()
    results = []
    for _ in range(n):
        g = Sokoban(level, max_steps=MAX_STEPS)
        e = MCTSEngine(g, iterations=ITERS, max_rollout_depth=MAX_DEPTH, logging=True)
        if fn is not None:
            e.set_tool(PHASE, fn)
        r = e.play_game()
        results.append(r)
    elapsed = time.time() - t0
    avg_ret = sum(r["returns"][0] if isinstance(r["returns"], list)
                  else r["returns"] for r in results) / n
    solve_rate = sum(1 for r in results if r.get("solved")) / n
    avg_steps = sum(r.get("steps", MAX_STEPS) for r in results) / n
    return avg_ret, solve_rate, avg_steps, results, elapsed


def get_baseline(level):
    """Return baseline info for a level, computing lazily if needed."""
    if level not in level_baselines:
        print(f"  Computing baseline for {level}…")
        avg, sr, steps, _, t = multi_eval(None, level, n=EVAL_RUNS)
        comp = composite_score(sr, avg)
        level_baselines[level] = {
            "avg_returns": avg,
            "solve_rate": sr,
            "avg_steps": steps,
            "eval_time": t,
            "composite": comp,
        }
        level_best_scores[level] = comp
        print(f"    {level}: composite={comp:.4f}, solve_rate={sr:.0%}, "
              f"avg_returns={avg:.4f} ({t:.1f}s)")
        # Check if baseline already masters this level
        check_mastery(level, sr, steps)
    return level_baselines[level]


def build_history(results, current_level):
    """Build a concise history string from previous iterations."""
    bl = get_baseline(current_level)
    # Compute aggregate best = average of all per-level bests
    if level_best_scores:
        agg_best = sum(level_best_scores.values()) / len(level_best_scores)
    else:
        agg_best = 0.0

    lines = [
        f"Current level: {current_level}",
        f"Baseline for {current_level} (default MCTS): composite={bl['composite']:.4f}, "
        f"solve_rate={bl['solve_rate']:.0%}, avg_returns={bl['avg_returns']:.4f}",
        f"Aggregate best (avg across {len(level_best_scores)} levels): {agg_best:.4f}",
        "",
        "Per-level best composites so far:",
    ]
    for lv in sorted(level_best_scores.keys()):
        bl_lv = level_baselines.get(lv, {})
        tag = " [MASTERED]" if lv in mastered_levels else ""
        lines.append(f"  {lv}: best={level_best_scores[lv]:.4f} "
                      f"(baseline={bl_lv.get('composite', 0):.4f}){tag}")
    lines += [
        "",
        f"Active levels (not yet mastered): {sorted(active_levels)}",
        f"Mastered levels: {sorted(mastered_levels)}" if mastered_levels else "",
        "",
        "SCORING: composite = 0.6 × solve_rate + 0.4 × avg_returns",
        "  → SOLVING the puzzle is MORE important than heuristic accuracy.",
        "",
        "STRATEGY: Make ONE small change per iteration. Build on the",
        "previous version — do NOT rewrite from scratch.",
        "",
        "Recent iterations:",
    ]
    for r in results:
        tag = f"solve_rate={r['solve_rate']:.0%}"
        status = ""
        if r.get("is_best"):
            status = " ★ BEST-for-level"
        elif r.get("adopted"):
            status = " ← accepted"
        else:
            status = " ✗ rejected"
        time_info = ""
        if r.get("eval_time") is not None:
            time_info = f", eval_time={r['eval_time']:.1f}s"
        lines.append(
            f"  Iter {r['iteration']} [{r['level']}]: composite={r['composite']:.4f}, "
            f"{tag}{time_info}, desc={r.get('description', 'n/a')}{status}"
        )
    return "\n".join(lines)


# --- Compute initial baseline for starting level ---
print(f"Starting level: {LEVEL}")
init_bl = get_baseline(LEVEL)
print(f"  Reject floor for {LEVEL}: {init_bl['composite'] * REJECT_THRESHOLD:.4f}")
print(f"  Active levels: {active_levels}")


for iteration in range(1, NUM_ITERS + 1):
    # --- Check if all levels mastered ---
    if not active_levels:
        print(f"\n🎉 All levels mastered after {iteration - 1} iterations! Stopping early.")
        break

    cur_level = LEVEL   # capture level for this iteration
    bl = get_baseline(cur_level)  # ensure baseline exists, retrieve it
    reject_floor = bl["composite"] * REJECT_THRESHOLD

    print(f"\n{'#'*60}")
    print(f"  ITERATION {iteration}/{NUM_ITERS}, LEVEL={cur_level}")
    print(f"  Baseline composite={bl['composite']:.4f}, reject_floor={reject_floor:.4f}")
    print(f"  Active levels: {len(active_levels)}/{len(LEVELS)}, mastered: {sorted(mastered_levels)}")
    print(f"{'#'*60}")

    # Capture cur_level by value via default argument
    state_factory = lambda _lv=cur_level: Sokoban(_lv, max_steps=MAX_STEPS).new_initial_state()

    # --- 1. Play with CURRENT tool → collect trace ---
    t_play_start = time.time()
    g = Sokoban(cur_level, max_steps=MAX_STEPS)
    eng = MCTSEngine(g, iterations=ITERS, max_rollout_depth=MAX_DEPTH, logging=True)
    if current_fn is not None:
        eng.set_tool(PHASE, current_fn)

    play_result = eng.play_game()
    t_play = time.time() - t_play_start
    play_trace  = play_result.get("log_file", "")
    tl          = eng.get_tool_source()
    p_ret = play_result["returns"]
    p_ret_val = p_ret[0] if isinstance(p_ret, list) else p_ret
    ptag = "SOLVED" if play_result.get("solved") else "UNSOLVED"
    print(f"  Play: {ptag} in {play_result.get('steps','?')} steps  "
          f"returns={p_ret_val:.4f}  ({t_play:.1f}s)")

    # --- 2. Build history context ---
    history = build_history(all_results[-history_window:], cur_level) if all_results else None

    # --- 3. Optimize (3-step: analysis → draft → critique) ---
    t_opt_start = time.time()
    result = opt.run(
        record_files=[play_trace] if play_trace else [],
        tool_list=tl,
        state_factory=state_factory,
        additional_context=history,
    )
    t_opt = time.time() - t_opt_start
    print(f"  Optimize: {t_opt:.1f}s")

    # --- 4. Multi-run evaluation on SAME level ---
    iter_record = {
        "iteration": iteration,
        "level": cur_level,
        "smoke_test": result["smoke_test"],
        "avg_returns": bl["avg_returns"],
        "solve_rate": 0.0,
        "composite": 0.0,
        "avg_steps": MAX_STEPS,
        "description": (result.get("parsed") or {}).get("description", ""),
        "error": result.get("error"),
        "adopted": False,
        "is_best": False,
        "play_time": t_play,
        "opt_time": t_opt,
        "eval_time": None,
    }

    fn = result.get("function")
    if fn is not None:
        avg_ret, solve_rate, avg_steps, eval_runs, eval_time = multi_eval(fn, cur_level, n=EVAL_RUNS)
        comp = composite_score(solve_rate, avg_ret)
        iter_record["avg_returns"] = avg_ret
        iter_record["solve_rate"] = solve_rate
        iter_record["composite"] = comp
        iter_record["avg_steps"] = avg_steps
        iter_record["eval_time"] = eval_time

        etag = f"solve_rate={solve_rate:.0%}"
        print(f"  Eval ({EVAL_RUNS} runs, {cur_level}): avg_returns={avg_ret:.4f}, {etag}, "
              f"composite={comp:.4f}, avg_steps={avg_steps:.0f}  ({eval_time:.1f}s)")

        # Check if this eval masters the level
        check_mastery(cur_level, solve_rate, avg_steps)

        # --- 5. Accept-unless-terrible (per-level comparison) ---
        prev_level_best = level_best_scores.get(cur_level, bl["composite"])

        if comp > prev_level_best:
            # New best for this level
            print(f"  ★ NEW BEST for {cur_level} (prev={prev_level_best:.4f}) — adopting")
            level_best_scores[cur_level] = comp
            best_fn = fn
            current_fn = fn
            iter_record["adopted"] = True
            iter_record["is_best"] = True
        elif comp >= reject_floor:
            # Not best, but not terrible — accept to keep iterating
            print(f"  → Accepted on {cur_level} (comp={comp:.4f} ≥ floor={reject_floor:.4f}, "
                  f"level_best={prev_level_best:.4f})")
            current_fn = fn
            iter_record["adopted"] = True
        else:
            # Catastrophically bad — reject, revert to best
            print(f"  ✗ Rejected on {cur_level} (comp={comp:.4f} < floor={reject_floor:.4f}) "
                  f"— reverting to best")
            current_fn = best_fn
    else:
        print(f"  Eval:  SKIPPED (smoke test failed or error)")
        if result.get("error"):
            print(f"         {result['error'][:120]}")

    total_iter_time = t_play + t_opt + (iter_record["eval_time"] or 0)
    print(f"  Iteration total: {total_iter_time:.1f}s")
    all_results.append(iter_record)

    # Pick a new random level from non-mastered pool
    if active_levels:
        LEVEL = random.choice(active_levels)
    else:
        print(f"\n🎉 All levels mastered! Will stop at next iteration.")
        LEVEL = random.choice(LEVELS)  # fallback, won't be used


# --- Summary ---
print(f"\n{'='*60}")
print(f"Iterative optimization complete — {len(all_results)} iterations done.")
if mastered_levels:
    print(f"Mastered {len(mastered_levels)}/{len(LEVELS)} levels: {sorted(mastered_levels)}")
if level_best_scores:
    agg = sum(level_best_scores.values()) / len(level_best_scores)
    print(f"Aggregate best composite (avg of {len(level_best_scores)} levels): {agg:.4f}")
    for lv in sorted(level_best_scores.keys()):
        bl_lv = level_baselines.get(lv, {})
        delta = level_best_scores[lv] - bl_lv.get("composite", 0)
        tag = " [MASTERED]" if lv in mastered_levels else ""
        print(f"  {lv}: best={level_best_scores[lv]:.4f} "
              f"(baseline={bl_lv.get('composite', 0):.4f}, Δ={delta:+.4f}){tag}")
print(f"{'='*60}")

Starting level: level3
  Computing baseline for level3…
    level3: composite=0.6667, solve_rate=67%, avg_returns=0.6667 (5.8s)
  Reject floor for level3: 0.3333
  Active levels: ['level1', 'level2', 'level3', 'level4', 'level5', 'level6', 'level7', 'level8', 'level9', 'level10']

############################################################
  ITERATION 1/5, LEVEL=level3
  Baseline composite=0.6667, reject_floor=0.3333
  Active levels: 10/10, mastered: []
############################################################
  Play: SOLVED in 74 steps  returns=1.0000  (1.0s)
Step 1/6: Building analysis prompt…
Step 2/6: Querying LLM (step 2 — draft code)…
Step 3/6: Querying LLM (step 3 — critique & finalize)…
  LLM query: status=success  elapsed=15.3s
  Step-1 analysis length: 2045 chars
Step 5/6: Parsing & validating response…
  Validation passed ✓
  Installed to: /Users/zhuyuezx/Documents/UCSD/Winter_2025/CSE291A/CSE291A_Project/Tool_Creation/MCTS_tools/simulation/simulation.py
Step 6/6: Smoke 

## Cell 4: Multi-Level Evaluation (level1–5)

Run both **baseline** (default MCTS) and **best LLM-optimized** function across all 5 Sokoban levels to measure generalization.

In [4]:
import time

LEVELS = ["level1", "level2", "level3", "level4", "level5", "level6", "level7", "level8", "level9", "level10"]
LEVEL_EVAL_RUNS = 3   # games per level per algorithm


def eval_on_level(level, fn, n=LEVEL_EVAL_RUNS):
    """Evaluate fn (or default if None) on a given level. Returns dict."""
    t0 = time.time()
    results = []
    for _ in range(n):
        g = Sokoban(level, max_steps=MAX_STEPS)
        e = MCTSEngine(g, iterations=ITERS, max_rollout_depth=MAX_DEPTH, logging=False)
        if fn is not None:
            e.set_tool(PHASE, fn)
        r = e.play_game()
        results.append(r)
    elapsed = time.time() - t0
    avg_ret = sum(r["returns"][0] if isinstance(r["returns"], list)
                  else r["returns"] for r in results) / n
    solve_rate = sum(1 for r in results if r.get("solved")) / n
    avg_steps = sum(r.get("steps", MAX_STEPS) for r in results) / n
    return {
        "level": level,
        "avg_returns": avg_ret,
        "solve_rate": solve_rate,
        "avg_steps": avg_steps,
        "time": elapsed,
    }


# ── Run evaluation across all levels ──
level_results = {"baseline": [], "optimized": []}

for lvl in LEVELS:
    print(f"Evaluating {lvl}…", end=" ", flush=True)
    # Baseline (no custom fn)
    rb = eval_on_level(lvl, None, n=LEVEL_EVAL_RUNS)
    level_results["baseline"].append(rb)
    # Optimized (best_fn, may be None → same as baseline)
    ro = eval_on_level(lvl, best_fn, n=LEVEL_EVAL_RUNS)
    level_results["optimized"].append(ro)
    print(f"baseline={rb['avg_returns']:.3f} ({rb['solve_rate']:.0%})  "
          f"optimized={ro['avg_returns']:.3f} ({ro['solve_rate']:.0%})  "
          f"[{rb['time']:.1f}s + {ro['time']:.1f}s]")

# ── Print summary table ──
print(f"\n{'Level':<10} {'Base Solve%':>12} {'Opt Solve%':>12} {'Base AvgRet':>12} {'Opt AvgRet':>12} {'Base Steps':>11} {'Opt Steps':>11}")
print("─" * 82)
for rb, ro in zip(level_results["baseline"], level_results["optimized"]):
    print(f"{rb['level']:<10} {rb['solve_rate']*100:>11.0f}% {ro['solve_rate']*100:>11.0f}% "
          f"{rb['avg_returns']:>12.3f} {ro['avg_returns']:>12.3f} "
          f"{rb['avg_steps']:>11.1f} {ro['avg_steps']:>11.1f}")

Evaluating level1… baseline=1.000 (100%)  optimized=1.000 (100%)  [0.0s + 0.0s]
Evaluating level2… baseline=1.000 (100%)  optimized=1.000 (100%)  [0.0s + 0.3s]
Evaluating level3… baseline=1.000 (100%)  optimized=1.000 (100%)  [3.3s + 21.6s]
Evaluating level4… baseline=1.000 (100%)  optimized=1.000 (100%)  [0.2s + 2.9s]
Evaluating level5… baseline=0.333 (33%)  optimized=1.000 (100%)  [7.0s + 42.3s]
Evaluating level6… baseline=0.000 (0%)  optimized=0.000 (0%)  [3.7s + 45.3s]
Evaluating level7… baseline=0.000 (0%)  optimized=0.000 (0%)  [7.4s + 65.4s]
Evaluating level8… baseline=0.000 (0%)  optimized=0.000 (0%)  [6.5s + 76.0s]
Evaluating level9… baseline=0.000 (0%)  optimized=0.333 (33%)  [8.1s + 68.0s]
Evaluating level10… baseline=0.000 (0%)  optimized=0.000 (0%)  [14.5s + 153.2s]

Level       Base Solve%   Opt Solve%  Base AvgRet   Opt AvgRet  Base Steps   Opt Steps
──────────────────────────────────────────────────────────────────────────────────
level1             100%         100%   